# Auditoría y Monitorización de Seguridad en LLMs

> Implementa un sistema de auditoría completo para registrar interacciones, detectar
> anomalías y generar dashboards de seguridad para sistemas LLM en producción.

## Objetivos
- Construir `AuditLogger` para registrar llamadas a LLMs con métricas
- Detectar anomalías: frecuencia alta, tokens excesivos, patrones repetidos
- Generar dashboard con matplotlib: timeline, distribuciones, alertas
- Exportar informe de auditoría estructurado

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic pandas matplotlib --quiet

## 2. Clase AuditLogger

In [ ]:
import anthropic
import pandas as pd
import matplotlib.pyplot as plt
import hashlib
import json
import time
import random
from datetime import datetime, timedelta

client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

class AuditLogger:
    """Registra y analiza todas las interacciones con LLMs."""

    def __init__(self):
        self.registros = []

    def registrar(self, usuario: str, input_texto: str, output_texto: str,
                  tokens_entrada: int, tokens_salida: int, latencia_ms: int,
                  bloqueado: bool = False, motivo_bloqueo: str = None):
        """Registra una interacción con hash del contenido para no exponer datos sensibles."""
        self.registros.append({
            "timestamp": datetime.now().isoformat(),
            "usuario": usuario,
            "input_hash": hashlib.sha256(input_texto.encode()).hexdigest()[:16],
            "input_longitud": len(input_texto),
            "output_hash": hashlib.sha256(output_texto.encode()).hexdigest()[:16],
            "tokens_entrada": tokens_entrada,
            "tokens_salida": tokens_salida,
            "latencia_ms": latencia_ms,
            "bloqueado": bloqueado,
            "motivo_bloqueo": motivo_bloqueo
        })

    def detectar_anomalias(self) -> list[dict]:
        """Detecta patrones sospechosos en el log de auditoría."""
        if not self.registros:
            return []

        df = pd.DataFrame(self.registros)
        alertas = []

        # 1. Usuarios con muchas peticiones en poco tiempo
        freq_usuario = df["usuario"].value_counts()
        for usuario, count in freq_usuario.items():
            if count > 10:
                alertas.append({"tipo": "ALTA_FRECUENCIA", "detalle": f"{usuario}: {count} peticiones", "severidad": "alta"})

        # 2. Peticiones con tokens excesivos (posible extracción de datos)
        umbral_tokens = df["tokens_salida"].quantile(0.95)
        excesivos = df[df["tokens_salida"] > umbral_tokens * 2]
        if len(excesivos) > 0:
            alertas.append({"tipo": "TOKENS_EXCESIVOS", "detalle": f"{len(excesivos)} peticiones con tokens > {umbral_tokens*2:.0f}", "severidad": "media"})

        # 3. Tasa de bloqueo alta
        tasa_bloqueo = df["bloqueado"].mean()
        if tasa_bloqueo > 0.2:
            alertas.append({"tipo": "ALTA_TASA_BLOQUEO", "detalle": f"Tasa de bloqueo: {tasa_bloqueo:.1%}", "severidad": "alta"})

        return alertas

logger = AuditLogger()
print("AuditLogger inicializado.")

## 3. Simular 20 interacciones (mezcla normal y sospechosa)

In [ ]:
random.seed(42)
usuarios = ["user_alice", "user_bob", "user_charlie", "bot_sospechoso", "user_diana"]

# Simular log de 20 interacciones con datos realistas
for i in range(20):
    # bot_sospechoso hace muchas peticiones
    if i % 3 == 0:
        usuario = "bot_sospechoso"
    else:
        usuario = random.choice(usuarios[:3] + [usuarios[4]])

    bloqueado = random.random() < 0.15  # 15% de tasa de bloqueo
    tokens_salida = random.randint(50, 300) if not bloqueado else 0
    if usuario == "bot_sospechoso" and not bloqueado and random.random() < 0.3:
        tokens_salida = random.randint(800, 1500)  # tokens excesivos

    logger.registrar(
        usuario=usuario,
        input_texto=f"Pregunta de prueba número {i}",
        output_texto=f"Respuesta número {i}" if not bloqueado else "",
        tokens_entrada=random.randint(20, 100),
        tokens_salida=tokens_salida,
        latencia_ms=random.randint(200, 2000),
        bloqueado=bloqueado,
        motivo_bloqueo="JAILBREAK" if bloqueado and random.random() < 0.5 else ("PROHIBITED" if bloqueado else None)
    )

df_audit = pd.DataFrame(logger.registros)
print(f"Simuladas {len(df_audit)} interacciones.")
print(f"Bloqueadas: {df_audit['bloqueado'].sum()} ({df_audit['bloqueado'].mean():.1%})")
print(f"Tokens totales generados: {df_audit['tokens_salida'].sum():,}")

## 4. Detección de anomalías

In [ ]:
alertas = logger.detectar_anomalias()
print(f"=== ALERTAS DE SEGURIDAD ({len(alertas)}) ===")
for alerta in alertas:
    icono = "🔴" if alerta["severidad"] == "alta" else "🟡"
    print(f"{icono} [{alerta['tipo']}] {alerta['detalle']}")

print("\n=== ACTIVIDAD POR USUARIO ===")
print(df_audit.groupby("usuario")[["tokens_salida", "bloqueado", "latencia_ms"]].agg({"tokens_salida": "sum", "bloqueado": "sum", "latencia_ms": "mean"}).round(1))

## 5. Dashboard de seguridad

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Dashboard de Auditoría de Seguridad LLM", fontsize=14, fontweight="bold")

# 1. Tokens por usuario
tokens_usuario = df_audit.groupby("usuario")["tokens_salida"].sum().sort_values(ascending=False)
colores = ["#e74c3c" if u == "bot_sospechoso" else "#3498db" for u in tokens_usuario.index]
axes[0, 0].bar(tokens_usuario.index, tokens_usuario.values, color=colores, edgecolor="black", alpha=0.8)
axes[0, 0].set_title("Tokens Totales por Usuario", fontweight="bold")
axes[0, 0].tick_params(axis="x", rotation=30)
axes[0, 0].set_ylabel("Tokens")

# 2. Distribución de latencias
axes[0, 1].hist(df_audit["latencia_ms"], bins=10, color="#9b59b6", edgecolor="black", alpha=0.8)
axes[0, 1].set_title("Distribución de Latencias", fontweight="bold")
axes[0, 1].set_xlabel("Latencia (ms)")
axes[0, 1].set_ylabel("Frecuencia")

# 3. Peticiones bloqueadas vs permitidas por usuario
pivot = df_audit.groupby(["usuario", "bloqueado"]).size().unstack(fill_value=0)
pivot.plot(kind="bar", ax=axes[1, 0], color=["#2ecc71", "#e74c3c"], edgecolor="black", alpha=0.8)
axes[1, 0].set_title("Peticiones Bloqueadas vs Permitidas", fontweight="bold")
axes[1, 0].tick_params(axis="x", rotation=30)
axes[1, 0].legend(["Permitidas", "Bloqueadas"])

# 4. Motivos de bloqueo
bloqueados = df_audit[df_audit["bloqueado"] == True]
if len(bloqueados) > 0:
    motivos = bloqueados["motivo_bloqueo"].value_counts()
    axes[1, 1].pie(motivos.values, labels=motivos.index, autopct="%1.0f%%",
                   colors=["#e74c3c", "#f39c12", "#e67e22"])
    axes[1, 1].set_title("Motivos de Bloqueo", fontweight="bold")

plt.tight_layout()
plt.savefig("auditoria_seguridad.png", dpi=150, bbox_inches="tight")
plt.show()

# Exportar informe
informe = {
    "generado": datetime.now().isoformat(),
    "total_peticiones": len(df_audit),
    "tasa_bloqueo": float(df_audit["bloqueado"].mean()),
    "tokens_totales": int(df_audit["tokens_salida"].sum()),
    "alertas": alertas
}
with open("informe_auditoria.json", "w", encoding="utf-8") as f:
    json.dump(informe, f, indent=2, ensure_ascii=False)
print("Informe exportado a 'informe_auditoria.json'")